# 최신 단일채널 화자 구분 ASR 비교

Nemotron-3-Diarization + Qwen3-ASR-1.7B, VibeVoice-ASR 9B, MOSS-Transcribe-Diarize 0.9B를 같은 평가셋에 실행합니다. 실제 입력은 **항상 마이크 하나**이며, proposed와 같은 기준 채널(활성 마이크 중 최소 번호)을 선택합니다.

먼저 `LIMIT=2`로 추론·재개·채점을 확인한 뒤 `LIMIT=0`으로 전체 평가하세요. A100을 권장하며, 특히 VibeVoice는 가중치만 약 18GB이므로 T4에 적합하지 않습니다. 이 노트북은 모델을 학습하지 않습니다.

세 기본 설정은 20초 전체를 관측합니다. `full_audio` 참고 비교로 보고하며, 실시간 proposed와 관측 범위를 함께 표시합니다. 우위 여부는 실행 결과로 판단합니다.


In [ ]:
STORAGE_BACKEND: str = "drive_api"  # @param ["drive_api", "mounted_drive"]
DRIVE_SECRET: str = "DRIVE_AUTH"  # @param {type:"string"}
LOCAL_STORE_ROOT: str = "/content/vcsasr-store"
BACKUP_EVERY_SEC: int = 120  # @param {type:"integer"}

BASELINES: list[str] = ["nemotron", "vibevoice", "moss"]
LIMIT: int = 2  # @param {type:"integer"}
MAX_NEW_TOKENS: int = 2048  # @param {type:"integer"}
BRANCH: str = "main"  # @param {type:"string"}
COMMIT: str = ""  # @param {type:"string"}
REPO_URL: str = "https://github.com/hd1534/variable-channel-spatial-asr.git"
REPO_DIR: str = "/content/variable-channel-spatial-asr"
DRIVE_ROOT: str = LOCAL_STORE_ROOT if STORAGE_BACKEND == "drive_api" else "/content/drive/MyDrive/vcsasr"
ENV_ROOT: str = "/content/vcsasr-envs"
SETS: list[str] = ["eval-t2", "eval-t2-static"]
RUN_TAG: str = ""  # @param {type:"string"}
ASR_CACHE_NAME: str = "asr_cache"  # @param {type:"string"}


## 개인 Drive 연결 (처음 한 번)

학교 계정의 Colab Pro+를 유지하고 개인 계정의 Drive를 **Google 공식 Python 라이브러리**로 사용합니다.
[개인 Drive 설정 가이드](https://github.com/hd1534/variable-channel-spatial-asr/blob/main/docs/personal-drive.md)에 따라
Google Cloud 프로젝트에서 Drive API와 Desktop OAuth 클라이언트를 만들고, Mac에서 `python3 setup_drive.py`를 실행하세요.
생성된 `runs/private/drive-auth.json` 내용을 **Colab 왼쪽 열쇠 아이콘 → 보안 비밀 → `DRIVE_AUTH`**에 저장하고 노트북 접근을 허용합니다.
인증 파일·토큰은 채팅, 코드 셀, 출력, Git에 붙여 넣지 않습니다. 기존 rclone 설정은 사용하지 않습니다.

`STORAGE_BACKEND="drive_api"`가 기본입니다. 학교 Drive를 마운트하지 않으며, 파일은 VM의 `LOCAL_STORE_ROOT`에서 읽고 씁니다.
재접속하면 마지막 완료 백업을 검증해 복원합니다. 같은 계정의 일반 Drive 마운트는 `mounted_drive`로 선택할 수 있습니다.
OAuth 앱이 **Testing** 상태이면 갱신 토큰이 7일 뒤 만료되므로, 장기 사용 전 가이드의 게시 상태 안내도 확인하세요.

같은 백업 폴더는 한 런타임에서만 사용합니다. 120초마다 학습 결과를 백업하고, 명령 종료 시 대용량 모델·코퍼스 캐시도 백업합니다.
업로드한 크기·체크섬을 확인한 뒤에만 완료 기록을 저장합니다. 전송 실패는 셀 오류로 표시합니다.
갑작스러운 런타임 종료 시 **마지막 완료 백업 이후 작업은 유실될 수 있습니다**. 마지막 셀의 백업 완료 메시지까지 확인하세요.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any

if "storage" not in globals():
    storage: Any | None = None
elif storage is not None:
    try:
        storage.close()
    except RuntimeError as exc:
        print(f"이전 백업 세션 종료: {exc}", flush=True)
    storage = None


def run_command(arguments: list[str]) -> None:
    """Run a command without shell interpolation and stop on failure."""
    print("$", " ".join(arguments), flush=True)
    if storage is not None:
        storage.run(arguments)
    else:
        subprocess.run(arguments, check=True)


if not Path(REPO_DIR, ".git").is_dir():
    run_command(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                 REPO_URL, REPO_DIR])
run_command(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/*", "!/data/eval-dev/audio/"])
run_command(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", COMMIT or BRANCH])
run_command(["git", "-C", REPO_DIR, "checkout", "FETCH_HEAD"])
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
run_command(["git", "log", "-1", "--format=%h %s"])

if STORAGE_BACKEND == "drive_api":
    from drive_storage import connect_personal_drive
    run_command([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements-drive.txt"])
    storage = connect_personal_drive(Path(LOCAL_STORE_ROOT),
                                     secret_name=DRIVE_SECRET, interval_sec=BACKUP_EVERY_SEC)
elif STORAGE_BACKEND == "mounted_drive":
    if "google.colab" in sys.modules:
        from google.colab import drive
        drive.mount("/content/drive")
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
else:
    raise ValueError("STORAGE_BACKEND를 확인하세요.")
print(f"저장소: {STORAGE_BACKEND} | 작업 경로: {DRIVE_ROOT}", flush=True)

Path(DRIVE_ROOT, "mono_baselines").mkdir(parents=True, exist_ok=True)

os.environ["VCSASR_MODEL_CACHE"] = str(Path(DRIVE_ROOT, "models", "asr"))
# Cache public Hugging Face weights without moving authentication tokens into the backup tree.
os.environ["HF_HUB_CACHE"] = str(Path(DRIVE_ROOT, "models", "huggingface"))


def run_name(name: str) -> str:
    """Select a separate output directory when experiment settings change."""
    return f"{name}-{RUN_TAG}" if RUN_TAG else name


## 모델별 환경과 GPU 확인

기존 학습 환경을 갱신하지 않고 별도 venv에 공식 구현을 설치합니다. 모델 가중치와 공식 소스는 코드에 기록된 revision을 사용합니다. 설치된 패키지 목록도 결과 폴더에 보존합니다. 설치와 GPU 모델 실행은 Colab에서 확인해야 합니다.


In [ ]:
import shutil
import torch
from setup_asr_env import setup

if not torch.cuda.is_available():
    raise RuntimeError("런타임 유형을 GPU로 바꾼 뒤 다시 실행하세요.")
print(torch.cuda.get_device_name(0), "| VRAM GB", torch.cuda.get_device_properties(0).total_memory / 1e9)
if LIMIT < 0 or MAX_NEW_TOKENS < 1:
    raise ValueError("LIMIT은 0 이상, MAX_NEW_TOKENS는 1 이상이어야 합니다.")
if not BASELINES or any(name not in ("nemotron", "vibevoice", "moss") for name in BASELINES):
    raise ValueError("BASELINES를 확인하세요.")
profiles: list[str] = list(dict.fromkeys(BASELINES + (["qwen"] if "nemotron" in BASELINES else [])))
interpreters: dict[str, str] = {name: str(setup(name, Path(ENV_ROOT))) for name in profiles}
for name in profiles:
    destination: Path = Path(DRIVE_ROOT, "mono_baselines", "runtimes", name)
    destination.mkdir(parents=True, exist_ok=True)
    for filename in ("resolved-requirements.txt", "vcsasr-runtime.json"):
        shutil.copy2(Path(ENV_ROOT, name, filename), destination / filename)


## 추론과 채점

완료한 세션은 건너뜁니다. 모델·디코딩 설정·오디오가 달라지면 기존 결과와 혼합하지 않고 중단하므로, 변경된 실험은 다른 출력 경로를 사용하세요. `LIMIT=2` 실행 후 `LIMIT=0`으로 바꾸면 나머지 세션을 이어서 처리합니다. 파싱 실패나 토큰 상한에 도달한 출력은 성공으로 저장하지 않습니다.

모델·토큰 상한을 바꿀 때에는 `RUN_TAG`로 새 실험 경로를 지정합니다. CPU/GPU 정밀도나 패키지 버전이 달라졌다면 `ASR_CACHE_NAME`도 새 이름으로 설정하고 학습 노트북과 맞추세요.


In [ ]:
for name in BASELINES:
    arguments: list[str] = [sys.executable, "mono_baselines.py", "--backend", name,
                            "--python", interpreters[name], "--out", str(Path(DRIVE_ROOT, "mono_baselines", run_name(name))),
                            "--asr-cache", str(Path(DRIVE_ROOT, ASR_CACHE_NAME)), "--device", "cuda",
                            "--limit", str(LIMIT), "--max-new-tokens", str(MAX_NEW_TOKENS), "--sets", *SETS]
    if name == "nemotron":
        arguments += ["--qwen-python", interpreters["qwen"]]
    run_command(arguments)


## 비교 결과

`paired M`은 상대 proposed가 사용한 마이크 수입니다. 아래 모든 baseline의 실제 입력은 **1ch**입니다. 같은 케이스끼리 SpkAcc와 cpWER를 비교하며, 동일 마이크의 출력을 서로 다른 M에서 반복 평가한 값을 독립 표본으로 간주하지 않습니다. 추후 신뢰구간을 계산할 때에는 세션 단위로 묶으세요.

Nemotron은 proposed와 같은 Qwen 전사·최소 1초 slot-run 규칙을 사용하고, VibeVoice/MOSS는 자체 전사를 사용합니다. 2인 평가 계약에 맞춰 예측 화자 수가 2명을 넘으면 발화 시간이 긴 두 화자를 유지하고 나머지는 최장 화자에 합칩니다. 이 후처리는 정답을 사용하지 않으며 결과에 정책을 기록합니다.


In [ ]:
reports: dict[str, dict] = {}
for name in BASELINES:
    report_path: Path = Path(DRIVE_ROOT, "mono_baselines", run_name(name), "score.json")
    reports[name] = json.loads(report_path.read_text(encoding="utf-8"))
for set_name in SETS:
    print(f"\n{set_name} | input=1ch | full_audio | LIMIT={LIMIT} (0=전체)")
    for name, report in reports.items():
        values: list[str] = []
        for m, metrics in report[set_name]["by_comparison_M"].items():
            values.append(f"paired M={m}: Acc {metrics['speaker_attribution_accuracy']:.3f}, cpWER {metrics['cpwer']:.3f}")
        print(report["baseline"]["run"], " | ".join(values))
print("===== VC-SASR MONO BASELINES =====")
print(json.dumps(reports, ensure_ascii=False))
print("===== END =====")


## Proposed와 함께 보기

학습 노트북에서 생성한 결과가 있으면 같은 Drive에서 비교표를 만듭니다. `input`은 실제 입력 채널 수이고 `scope`는 관측 범위입니다. 다른 평가 부분집합은 별도 population으로 표시합니다.


In [ ]:
run_command([sys.executable, "compare_asr.py", "--eval-root", str(Path(DRIVE_ROOT, "eval")),
             "--mono-root", str(Path(DRIVE_ROOT, "mono_baselines")),
             "--out", str(Path(DRIVE_ROOT, "asr_comparison.json")), "--sets", *SETS])

if storage is not None:
    storage.flush()
    print("개인 Drive 최종 백업 완료", flush=True)
